In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(r"C:\Users\Ameet Computer\Music\ML engineer projects\Ecommerce ML project\eCommercePK.csv")
print(df.head(5))

   order_id order_status order_source  order_date category      sku  quantity  \
0      5447      Shipped     Whatsapp  04/01/2025       CK      Dng         1   
1     14127      Shipped          Web  08/01/2025       CK      DIY         1   
2     14213      Shipped          Web  19/01/2025       CK      DIY         1   
3     14333      Shipped          Web  05/02/2025       CK  Simply9         1   
4     14397      Shipped          Web  16/02/2025       CK  Simply9         1   

   sales     city  
0   1590  Karachi  
1    899  Karachi  
2    899  Karachi  
3    990  Karachi  
4    990  Karachi  


order status is extra column not useful for prediction and may data leakage 

In [3]:
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
792    False
793    False
794    False
795    False
796    False
Length: 797, dtype: bool

In [4]:
df.isnull().sum()

order_id        0
order_status    0
order_source    0
order_date      0
category        0
sku             0
quantity        0
sales           0
city            0
dtype: int64

In [6]:
df.describe()

,order_id,quantity,sales
count,7.970000e+02,797.000000,797.000000
mean,1.666269e+04,1.058971,1925.685069
std,5.075915e+04,0.422704,1093.164198
min,5.444000e+03,1.000000,899.000000
25%,1.423100e+04,1.000000,1390.000000
50%,1.441200e+04,1.000000,1590.000000
75%,1.464000e+04,1.000000,2190.000000
max,1.413666e+06,8.000000,13500.000000


In [7]:
df.dtypes

order_id         int64
order_status    object
order_source    object
order_date      object
category        object
sku             object
quantity         int64
sales            int64
city            object
dtype: object

In [8]:
df.shape

(797, 9)

In [12]:
columns = df.select_dtypes(include='object').columns.tolist()
df[columns] = df[columns].apply(lambda x: x.str.strip())

In [14]:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="%d/%m/%Y"
)

In [15]:
print(df["order_date"].head())

0   2025-01-04
1   2025-01-08
2   2025-01-19
3   2025-02-05
4   2025-02-16
Name: order_date, dtype: datetime64[ns]


In [16]:
####datetime featres
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["day"] = df["order_date"].dt.day
df["day_of_week"] = df["order_date"].dt.dayofweek

In [17]:
df = df.drop("order_date", axis=1)

In [18]:
x = df.drop("sales",axis=1)
y = df["sales"]

In [19]:
x.head(5)

,order_id,order_status,order_source,category,sku,quantity,city,year,month,day,day_of_week
0,5447,Shipped,Whatsapp,CK,Dng,1,Karachi,2025,1,4,5
1,14127,Shipped,Web,CK,DIY,1,Karachi,2025,1,8,2
2,14213,Shipped,Web,CK,DIY,1,Karachi,2025,1,19,6
3,14333,Shipped,Web,CK,Simply9,1,Karachi,2025,2,5,2
4,14397,Shipped,Web,CK,Simply9,1,Karachi,2025,2,16,6


In [20]:
y.head(5)

0    1590
1     899
2     899
3     990
4     990
Name: sales, dtype: int64

In [22]:
x= x.drop("order_id",axis=1)

In [23]:
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42)

In [24]:
df.head(5)

,order_status,order_source,category,sku,quantity,sales,city,year,month,day,day_of_week
0,Shipped,Whatsapp,CK,Dng,1,1590,Karachi,2025,1,4,5
1,Shipped,Web,CK,DIY,1,899,Karachi,2025,1,8,2
2,Shipped,Web,CK,DIY,1,899,Karachi,2025,1,19,6
3,Shipped,Web,CK,Simply9,1,990,Karachi,2025,2,5,2
4,Shipped,Web,CK,Simply9,1,990,Karachi,2025,2,16,6


order source,category sku city cartegorical 
quanitity year month day day of week numerical

In [25]:
df = df.drop("order_status",axis=1)

In [26]:
x = x.drop("order_status",axis=1)

In [27]:
numeric_features = [
    "quantity",
    "year",
    "month",
    "day",
    "day_of_week"
]

In [29]:
categorical_features = [
    "order_source",
    "category",
    "sku",
    "city"
]

In [30]:
##Build numerical pipeline
numerical_pipeline = Pipeline  ([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

In [31]:
###Build Categorical column
categorical_pipeline = Pipeline ([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    "encoder",OneHotEncoder(handle_unknown="ignore")
])

In [32]:
###combine both pipelines 
preprocessor = ColumnTransformer([
    ("numerical",numerical_pipeline,numeric_features),
    ("categorical",categorical_pipeline,categorical_features)
])  

In [33]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    RandomForestRegressor,
    AdaBoostRegressor
)

In [34]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [37]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=0.5),
    "Lasso": Lasso(alpha=0.5),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

In [38]:
for name,model in models.items():
    Pipeline = Pipeline([
        ("preprocessor",preprocessor),
        ("model",model)
    ])
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)
    
    rmse = np.sqrt(mean_squared_error(y_test,y_pred))
    mae = mean_absolute_error(y_test,y_pred)
    r2 = r2_score(y_test,y_pred)
    print(name)
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)

NameError: name 'pipeline' is not defined